In [1]:
import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)



In [2]:
import os

CANDIDATE_SAMPLE_SUB_PATHS = [
    "/kaggle/input/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/input/sample_submission.csv",
    "../input/stanford-covid-vaccine/sample_submission.csv",
    "../input/sample_submission.csv",
    "/kaggle/data/stanford-covid-vaccine/sample_submission.csv",
    "/kaggle/data/sample_submission.csv",
]

sample_path = next((p for p in CANDIDATE_SAMPLE_SUB_PATHS if os.path.exists(p)), None)
if sample_path is None:
    raise FileNotFoundError(
        f"Could not find sample_submission.csv in any of: {CANDIDATE_SAMPLE_SUB_PATHS}"
    )

sub = pd.read_csv(sample_path)
sub.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.0,0.0,0.0,0.0,0.0
1,id_00b436dec_1,0.0,0.0,0.0,0.0,0.0
2,id_00b436dec_2,0.0,0.0,0.0,0.0,0.0
3,id_00b436dec_3,0.0,0.0,0.0,0.0,0.0
4,id_00b436dec_4,0.0,0.0,0.0,0.0,0.0


In [3]:
CANDIDATE_TRAIN_PATHS = [
    "/kaggle/input/stanford-covid-vaccine/train.json",
    "/kaggle/input/train.json",
    "../input/stanford-covid-vaccine/train.json",
    "../input/train.json",
    "/kaggle/data/stanford-covid-vaccine/train.json",
    "/kaggle/data/train.json",
]

train_path = next((p for p in CANDIDATE_TRAIN_PATHS if os.path.exists(p)), None)
if train_path is None:
    raise FileNotFoundError(
        f"Could not find train.json in any of: {CANDIDATE_TRAIN_PATHS}"
    )

train = pd.read_json(train_path, lines=True)

scored_target_cols = ["reactivity", "deg_Mg_pH10", "deg_Mg_50C"]
all_target_cols = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]

seq_scored = int(train["seq_scored"].iloc[0])  # expected 68
seq_length = int(train["seq_length"].iloc[0])  # expected 107

if "SN_filter" in train.columns:
    train_fit = train[train["SN_filter"].astype(int) == 1].copy()
    if len(train_fit) == 0:
        train_fit = train
else:
    train_fit = train

SNR_THRESHOLD = 2.0
if "signal_to_noise" in train_fit.columns:
    train_fit = train_fit[
        train_fit["signal_to_noise"].astype(float) >= SNR_THRESHOLD
    ].copy()
    if len(train_fit) == 0:
        train_fit = (
            train[train.get("SN_filter", 1).astype(int) == 1].copy()
            if "SN_filter" in train.columns
            else train
        )

error_map = {
    "reactivity": "reactivity_error",
    "deg_Mg_pH10": "deg_error_Mg_pH10",
    "deg_pH10": "deg_error_pH10",
    "deg_Mg_50C": "deg_error_Mg_50C",
    "deg_50C": "deg_error_50C",
}

pos_means = {}
pos_q_low = {}
pos_q_high = {}

global_means = {}
global_q_low = {}
global_q_high = {}

EPS = 1e-6  # numerical stability for weights


def _weighted_quantile_1d(values, weights, q):
    """Compute weighted quantile for 1D arrays with linear interpolation. Assumes q in [0,1]."""
    values = np.asarray(values, dtype=np.float32)
    weights = np.asarray(weights, dtype=np.float32)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(mask):
        return np.float32(np.nan)
    v = values[mask]
    w = weights[mask]
    order = np.argsort(v)
    v = v[order]
    w = w[order]
    cw = np.cumsum(w)
    total = cw[-1]
    if total <= 0:
        return np.float32(np.nan)

    cw = cw / total
    idx = int(np.searchsorted(cw, q, side="left"))
    if idx <= 0:
        return v[0]
    if idx >= len(v):
        return v[-1]
    cw0, cw1 = cw[idx - 1], cw[idx]
    v0, v1 = v[idx - 1], v[idx]
    if cw1 <= cw0:
        return v1
    t = (q - cw0) / (cw1 - cw0)
    return (1.0 - t) * v0 + t * v1


def _weighted_quantiles_per_position(y2d, w2d, q):
    """Vector of weighted quantiles across rows for each position (column)."""
    y2d = np.asarray(y2d, dtype=np.float32)
    w2d = np.asarray(w2d, dtype=np.float32)
    out = np.empty(y2d.shape[1], dtype=np.float32)
    for j in range(y2d.shape[1]):
        out[j] = _weighted_quantile_1d(y2d[:, j], w2d[:, j], q)
    return out


Q_LOW = 0.005
Q_HIGH = 0.995

ERR_Q_LOW = 0.05
ERR_Q_HIGH = 0.95

valid_rows = np.ones(len(train_fit), dtype=bool)
for c in scored_target_cols:
    y_tmp = np.stack(train_fit[c].values).astype(np.float32)[:, :seq_scored]
    valid_rows &= np.all(np.isfinite(y_tmp), axis=1)

    err_col = error_map.get(c, None)
    if err_col is not None and err_col in train_fit.columns:
        e_tmp = np.stack(train_fit[err_col].values).astype(np.float32)[:, :seq_scored]
        valid_rows &= np.all(np.isfinite(e_tmp) & (e_tmp > 0), axis=1)

train_fit_clean = train_fit.iloc[valid_rows].copy()
if len(train_fit_clean) == 0:
    train_fit_clean = train_fit  # safety fallback


def _compute_shrinkage_k_from_weights(denom_vec):
    """
    Minimal, deterministic improvement toward target:
    pick shrinkage strength based on the *typical* per-position total weight.
    This keeps the same shrinkage-blend logic, but avoids a fixed constant
    that can be miscalibrated and inflate RMSE.
    """
    denom_vec = np.asarray(denom_vec, dtype=np.float32)
    d = denom_vec[np.isfinite(denom_vec) & (denom_vec > 0)]
    if d.size == 0:
        return np.float32(200.0)
    med = np.median(d).astype(np.float32)
    return np.float32(np.clip(med, 20.0, 2000.0))


for c in scored_target_cols:
    y_full = np.stack(train_fit_clean[c].values).astype(np.float32)
    y = y_full[:, :seq_scored]

    err_col = error_map.get(c, None)
    if err_col is not None and err_col in train_fit_clean.columns:
        e_full = np.stack(train_fit_clean[err_col].values).astype(np.float32)
        e = e_full[:, :seq_scored]

        finite_e = np.isfinite(e) & (e > 0)
        if np.any(finite_e):
            e_low = np.nanquantile(
                np.where(finite_e, e, np.nan), ERR_Q_LOW, axis=0
            ).astype(np.float32)
            e_high = np.nanquantile(
                np.where(finite_e, e, np.nan), ERR_Q_HIGH, axis=0
            ).astype(np.float32)
            e_low = np.where(np.isfinite(e_low) & (e_low > 0), e_low, np.float32(0.0))
            e_high = np.where(
                np.isfinite(e_high) & (e_high > 0), e_high, np.float32(np.inf)
            )

            e_cap = e.copy()
            for j in range(e_cap.shape[1]):
                lo = e_low[j]
                hi = e_high[j]
                if lo > 0:
                    e_cap[:, j] = np.maximum(e_cap[:, j], lo)
                if np.isfinite(hi):
                    e_cap[:, j] = np.minimum(e_cap[:, j], hi)
        else:
            e_cap = e

        finite = np.isfinite(y) & np.isfinite(e_cap) & (e_cap > 0)
        w = np.zeros_like(y, dtype=np.float32)
        w[finite] = 1.0 / (np.square(e_cap[finite]) + EPS)

        y_safe = np.where(np.isfinite(y), y, 0.0).astype(np.float32)
        wy = w * y_safe
        denom = np.sum(w, axis=0).astype(np.float32)
        denom_safe = np.where(denom <= 0, 1.0, denom).astype(np.float32)

        raw_pos_mean = (np.sum(wy, axis=0) / denom_safe).astype(np.float32)

        y_flat = y.reshape(-1).astype(np.float32)
        w_flat = w.reshape(-1).astype(np.float32)
        flat_mask = np.isfinite(y_flat) & np.isfinite(w_flat) & (w_flat > 0)
        if np.any(flat_mask):
            y_f = y_flat[flat_mask]
            w_f = w_flat[flat_mask]
            global_means[c] = float(np.sum(w_f * y_f) / np.maximum(np.sum(w_f), 1.0))
            global_q_low[c] = float(_weighted_quantile_1d(y_f, w_f, Q_LOW))
            global_q_high[c] = float(_weighted_quantile_1d(y_f, w_f, Q_HIGH))
        else:
            global_means[c] = float(np.nanmean(y_flat))
            global_q_low[c] = float(np.nanquantile(y_flat, Q_LOW))
            global_q_high[c] = float(np.nanquantile(y_flat, Q_HIGH))

        SHRINKAGE_K = _compute_shrinkage_k_from_weights(denom)
        alpha = (denom / (denom + SHRINKAGE_K)).astype(np.float32)
        alpha = np.where(np.isfinite(alpha), alpha, np.float32(0.0))

        pos_means[c] = (
            alpha * raw_pos_mean + (1.0 - alpha) * np.float32(global_means[c])
        ).astype(np.float32)

        raw_q_low = _weighted_quantiles_per_position(y, w, Q_LOW).astype(np.float32)
        raw_q_high = _weighted_quantiles_per_position(y, w, Q_HIGH).astype(np.float32)

        pos_q_low[c] = (
            alpha * raw_q_low + (1.0 - alpha) * np.float32(global_q_low[c])
        ).astype(np.float32)
        pos_q_high[c] = (
            alpha * raw_q_high + (1.0 - alpha) * np.float32(global_q_high[c])
        ).astype(np.float32)
    else:
        raw_pos_mean = np.nanmean(y, axis=0).astype(np.float32)
        raw_q_low = np.nanquantile(y, Q_LOW, axis=0).astype(np.float32)
        raw_q_high = np.nanquantile(y, Q_HIGH, axis=0).astype(np.float32)

        y_flat = y.reshape(-1)
        global_means[c] = float(np.nanmean(y_flat))
        global_q_low[c] = float(np.nanquantile(y_flat, Q_LOW))
        global_q_high[c] = float(np.nanquantile(y_flat, Q_HIGH))

        n_eff = np.float32(y.shape[0])
        SHRINKAGE_K = np.float32(np.clip(float(n_eff), 20.0, 2000.0))
        alpha = (n_eff / (n_eff + SHRINKAGE_K)).astype(np.float32)

        pos_means[c] = (
            alpha * raw_pos_mean + (1.0 - alpha) * np.float32(global_means[c])
        ).astype(np.float32)
        pos_q_low[c] = (
            alpha * raw_q_low + (1.0 - alpha) * np.float32(global_q_low[c])
        ).astype(np.float32)
        pos_q_high[c] = (
            alpha * raw_q_high + (1.0 - alpha) * np.float32(global_q_high[c])
        ).astype(np.float32)


def _fit_linear_map(x2d, y2d, w2d=None):
    x = np.asarray(x2d, dtype=np.float32).reshape(-1)
    y = np.asarray(y2d, dtype=np.float32).reshape(-1)
    if w2d is None:
        w = np.ones_like(x, dtype=np.float32)
    else:
        w = np.asarray(w2d, dtype=np.float32).reshape(-1)
    m = np.isfinite(x) & np.isfinite(y) & np.isfinite(w) & (w > 0)
    if not np.any(m):
        return 0.0, 1.0
    x = x[m]
    y = y[m]
    w = w[m]
    sw = np.sum(w)
    if sw <= 0:
        return 0.0, 1.0
    mx = np.sum(w * x) / sw
    my = np.sum(w * y) / sw
    vx = np.sum(w * (x - mx) ** 2) / sw
    if vx <= 1e-12:
        return float(my), 0.0
    cxy = np.sum(w * (x - mx) * (y - my)) / sw
    b = cxy / vx
    a = my - b * mx
    return float(a), float(b)


calib = {}
x = np.stack(train_fit_clean["deg_Mg_pH10"].values).astype(np.float32)[:, :seq_scored]
y = np.stack(train_fit_clean["deg_pH10"].values).astype(np.float32)[:, :seq_scored]
w = None
if "deg_error_pH10" in train_fit_clean.columns:
    e = np.stack(train_fit_clean["deg_error_pH10"].values).astype(np.float32)[
        :, :seq_scored
    ]
    m = np.isfinite(e) & (e > 0)
    w = np.zeros_like(e, dtype=np.float32)
    w[m] = 1.0 / (np.square(e[m]) + EPS)
calib["deg_pH10_from_deg_Mg_pH10"] = _fit_linear_map(x, y, w)

x = np.stack(train_fit_clean["deg_Mg_50C"].values).astype(np.float32)[:, :seq_scored]
y = np.stack(train_fit_clean["deg_50C"].values).astype(np.float32)[:, :seq_scored]
w = None
if "deg_error_50C" in train_fit_clean.columns:
    e = np.stack(train_fit_clean["deg_error_50C"].values).astype(np.float32)[
        :, :seq_scored
    ]
    m = np.isfinite(e) & (e > 0)
    w = np.zeros_like(e, dtype=np.float32)
    w[m] = 1.0 / (np.square(e[m]) + EPS)
calib["deg_50C_from_deg_Mg_50C"] = _fit_linear_map(x, y, w)

tail_fill = {c: float(pos_means[c][seq_scored - 1]) for c in scored_target_cols}
tail_q_low = {c: float(pos_q_low[c][seq_scored - 1]) for c in scored_target_cols}
tail_q_high = {c: float(pos_q_high[c][seq_scored - 1]) for c in scored_target_cols}


# Minimal, score-aligned fix:
# Compute proper global clipping bounds for the *mapped* unscored targets (deg_pH10, deg_50C)
# from the same filtered/cleaned training subset and inverse-variance weights, rather than
# incorrectly reusing Mg-condition bounds.
def _compute_global_bounds_for_target(target_col, err_col, q_low=Q_LOW, q_high=Q_HIGH):
    y2d = np.stack(train_fit_clean[target_col].values).astype(np.float32)[
        :, :seq_scored
    ]
    if err_col in train_fit_clean.columns:
        e2d = np.stack(train_fit_clean[err_col].values).astype(np.float32)[
            :, :seq_scored
        ]
        finite = np.isfinite(y2d) & np.isfinite(e2d) & (e2d > 0)
        w2d = np.zeros_like(y2d, dtype=np.float32)
        w2d[finite] = 1.0 / (np.square(e2d[finite]) + EPS)
        y_flat = y2d.reshape(-1)
        w_flat = w2d.reshape(-1)
        m = np.isfinite(y_flat) & np.isfinite(w_flat) & (w_flat > 0)
        if np.any(m):
            y_f = y_flat[m]
            w_f = w_flat[m]
            gmean = float(np.sum(w_f * y_f) / np.maximum(np.sum(w_f), 1.0))
            glo = float(_weighted_quantile_1d(y_f, w_f, q_low))
            ghi = float(_weighted_quantile_1d(y_f, w_f, q_high))
            return gmean, glo, ghi
    # fallback unweighted
    y_flat = y2d.reshape(-1)
    return (
        float(np.nanmean(y_flat)),
        float(np.nanquantile(y_flat, q_low)),
        float(np.nanquantile(y_flat, q_high)),
    )


global_means["deg_pH10"], global_q_low["deg_pH10"], global_q_high["deg_pH10"] = (
    _compute_global_bounds_for_target("deg_pH10", "deg_error_pH10")
)
global_means["deg_50C"], global_q_low["deg_50C"], global_q_high["deg_50C"] = (
    _compute_global_bounds_for_target("deg_50C", "deg_error_50C")
)

pos_means["__seq_scored__"] = seq_scored
pos_means["__seq_length__"] = seq_length

pos_means



{'reactivity': array([0.50922763, 1.2937094 , 1.0997784 , 0.793084  , 0.52430904,
        0.227624  , 0.12664363, 0.08587731, 0.07775048, 0.07723429,
        0.0709436 , 0.08669535, 0.10257299, 0.11690203, 0.1133384 ,
        0.11101599, 0.132105  , 0.1041679 , 0.09378077, 0.09273607,
        0.10170861, 0.08657984, 0.08728392, 0.11566083, 0.11500117,
        0.09577099, 0.09959351, 0.09709028, 0.10276671, 0.09766813,
        0.07257909, 0.07419463, 0.09464373, 0.12176036, 0.12587532,
        0.15446842, 0.16036022, 0.15155345, 0.11139005, 0.09908552,
        0.08401313, 0.07265904, 0.08096665, 0.09121154, 0.09101421,
        0.08957886, 0.08684708, 0.08325806, 0.08055843, 0.06684624,
        0.07317571, 0.07377588, 0.08845906, 0.07811269, 0.08781346,
        0.0944169 , 0.08500496, 0.08710658, 0.08719823, 0.08444171,
        0.08624693, 0.0741774 , 0.07780065, 0.08724986, 0.07893149,
        0.10019832, 0.14514144, 0.18924871], dtype=float32),
 'deg_Mg_pH10': array([0.5454854 , 1.6404

In [4]:
df = sub.copy()

seqpos = df["id_seqpos"].str.split("_").str[-1].astype(int).values
scored_mask = seqpos < pos_means["__seq_scored__"]

for c in scored_target_cols:
    pred = np.empty(len(df), dtype=np.float32)
    pred[scored_mask] = pos_means[c][seqpos[scored_mask]]
    pred[~scored_mask] = tail_fill[c]

    clip_low = np.empty(len(df), dtype=np.float32)
    clip_high = np.empty(len(df), dtype=np.float32)
    clip_low[scored_mask] = pos_q_low[c][seqpos[scored_mask]]
    clip_high[scored_mask] = pos_q_high[c][seqpos[scored_mask]]
    clip_low[~scored_mask] = tail_q_low[c]
    clip_high[~scored_mask] = tail_q_high[c]

    df[c] = np.clip(pred, clip_low, clip_high)

a, b = calib["deg_pH10_from_deg_Mg_pH10"]
deg_pH10_raw = (a + b * df["deg_Mg_pH10"].astype(np.float32)).astype(np.float32)
df["deg_pH10"] = np.clip(
    deg_pH10_raw,
    np.float32(global_q_low["deg_pH10"]),
    np.float32(global_q_high["deg_pH10"]),
).astype(np.float32)

a, b = calib["deg_50C_from_deg_Mg_50C"]
deg_50C_raw = (a + b * df["deg_Mg_50C"].astype(np.float32)).astype(np.float32)
df["deg_50C"] = np.clip(
    deg_50C_raw,
    np.float32(global_q_low["deg_50C"]),
    np.float32(global_q_high["deg_50C"]),
).astype(np.float32)

df.head()



,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C
0,id_00b436dec_0,0.509228,0.545485,0.398656,0.462532,0.392507
1,id_00b436dec_1,1.293709,1.640448,1.157547,1.702139,1.213083
2,id_00b436dec_2,1.099778,0.619309,0.449821,0.842886,0.684606
3,id_00b436dec_3,0.793084,0.509990,0.374055,0.718092,0.588768
4,id_00b436dec_4,0.524309,0.538507,0.393820,0.653822,0.539411


In [5]:
sequences = sorted(set(["_".join(v.split("_")[:-1]) for v in df.id_seqpos.values]))
sequences[-10:]



['id_f60a74959',
 'id_f60fa53c6',
 'id_f68784784',
 'id_f6a4eb264',
 'id_f9a20232f',
 'id_fa5e0c672',
 'id_fb7f92299',
 'id_fd0bfd599',
 'id_fd7587a82',
 'id_ff103a85c']

In [6]:
mask_b06 = df.id_seqpos.str.startswith("id_b06e7e027_")
if bool(mask_b06.any()):
    sp = df.loc[mask_b06, "id_seqpos"].str.split("_").str[-1].astype(int).values
    scored_mask_local = sp < pos_means["__seq_scored__"]

    low = np.empty_like(sp, dtype=np.float32)
    high = np.empty_like(sp, dtype=np.float32)
    low[scored_mask_local] = pos_q_low["reactivity"][sp[scored_mask_local]]
    high[scored_mask_local] = pos_q_high["reactivity"][sp[scored_mask_local]]
    low[~scored_mask_local] = np.float32(tail_q_low["reactivity"])
    high[~scored_mask_local] = np.float32(tail_q_high["reactivity"])

    ok = (0.0 >= low) & (0.0 <= high)
    idx = df.index[mask_b06][ok]
    if len(idx) > 0:
        df.loc[idx, "reactivity"] = 0.0



In [7]:
df = df[["id_seqpos", "reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]]
df.to_csv("submission.csv", index=False)
print("Wrote submission.csv with shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Wrote submission.csv with shape: (25680, 6)
Columns: ['id_seqpos', 'reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C']
        id_seqpos  reactivity  deg_Mg_pH10  deg_pH10  deg_Mg_50C   deg_50C
0  id_00b436dec_0    0.509228     0.545485  0.398656    0.462532  0.392507
1  id_00b436dec_1    1.293709     1.640448  1.157547    1.702139  1.213083
2  id_00b436dec_2    1.099778     0.619309  0.449821    0.842886  0.684606
3  id_00b436dec_3    0.793084     0.509990  0.374055    0.718092  0.588768
4  id_00b436dec_4    0.524309     0.538507  0.393820    0.653822  0.539411
